In [28]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score

In [21]:
# Load the dataset
data = pd.read_csv('data/laptop_data.csv')
data.head(5)

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [22]:
# Feature Eengineering

data['Ram'] = data['Ram'].str.replace('GB','').astype(int)
data['Weight'] = data['Weight'].str.replace('kg','').astype(float)
data['Touchscreen'] = data['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)
data['Ips'] = data['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)
data['Cpu_Brand'] = data['Cpu'].apply(lambda x: x.split()[0])
data['Gpu_Brand'] = data['Gpu'].apply(lambda x: x.split()[0])
data['Cpu_frequency'] = data['Cpu'].apply(lambda x: float(x.split()[-1][:-3]) if x.split()[-1][-3:] == 'GHz' else float(x.split()[-1][:-3])/1000)
data['Has_HDD'] = data['Memory'].apply(lambda x: 1 if 'HDD' in x else 0)
data['Has_SSD'] = data['Memory'].apply(lambda x: 1 if 'SSD' in x else 0)
data['Has_Flash_Storage'] = data['Memory'].apply(lambda x: 1 if 'Flash Storage' in x else 0)

In [23]:
mem = data['Memory'].astype(str).str.lower()
mem = mem.str.replace(r'\.0', '', regex=True) # Fixes entries like "1.0TB" -> "1TB"
mem = mem.str.replace('tb', '000', regex=False) # Converts "1tb" to "1000"
mem = mem.str.replace('gb', '', regex=False)    # Removes "gb" so we are just left with numbers

# 3. Extract the exact numbers using Regex
data['SSD_GB'] = mem.str.extract(r'(\d+)\s*ssd', expand=False).fillna(0).astype(int)
data['HDD_GB'] = mem.str.extract(r'(\d+)\s*hdd', expand=False).fillna(0).astype(int)
data['Flash_GB'] = mem.str.extract(r'(\d+)\s*flash', expand=False).fillna(0).astype(int)


data = data.drop(columns=['Memory', 'ScreenResolution', 'Cpu', 'Gpu'])

In [24]:
print(data.head(5))

   Unnamed: 0 Company   TypeName  Inches  Ram  OpSys  Weight        Price  \
0           0   Apple  Ultrabook    13.3    8  macOS    1.37   71378.6832   
1           1   Apple  Ultrabook    13.3    8  macOS    1.34   47895.5232   
2           2      HP   Notebook    15.6    8  No OS    1.86   30636.0000   
3           3   Apple  Ultrabook    15.4   16  macOS    1.83  135195.3360   
4           4   Apple  Ultrabook    13.3    8  macOS    1.37   96095.8080   

   Touchscreen  Ips Cpu_Brand Gpu_Brand  Cpu_frequency  Has_HDD  Has_SSD  \
0            0    1     Intel     Intel            2.3        0        1   
1            0    0     Intel     Intel            1.8        0        0   
2            0    0     Intel     Intel            2.5        0        1   
3            0    1     Intel       AMD            2.7        0        1   
4            0    1     Intel     Intel            3.1        0        1   

   Has_Flash_Storage  SSD_GB  HDD_GB  Flash_GB  
0                  0     128   

In [25]:
X = data.drop('Price',axis=1)
y = data['Price']
print(X.head(5))
print(y.head(5))

   Unnamed: 0 Company   TypeName  Inches  Ram  OpSys  Weight  Touchscreen  \
0           0   Apple  Ultrabook    13.3    8  macOS    1.37            0   
1           1   Apple  Ultrabook    13.3    8  macOS    1.34            0   
2           2      HP   Notebook    15.6    8  No OS    1.86            0   
3           3   Apple  Ultrabook    15.4   16  macOS    1.83            0   
4           4   Apple  Ultrabook    13.3    8  macOS    1.37            0   

   Ips Cpu_Brand Gpu_Brand  Cpu_frequency  Has_HDD  Has_SSD  \
0    1     Intel     Intel            2.3        0        1   
1    0     Intel     Intel            1.8        0        0   
2    0     Intel     Intel            2.5        0        1   
3    1     Intel       AMD            2.7        0        1   
4    1     Intel     Intel            3.1        0        1   

   Has_Flash_Storage  SSD_GB  HDD_GB  Flash_GB  
0                  0     128       0         0  
1                  1       0       0       128  
2          

In [26]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [39]:
numeric_features = ['Inches', 'Weight','Touchscreen','Ips','Ram', 'Weight', 'Cpu_frequency', 'SSD_GB', 'HDD_GB', 'Flash_GB','Has_HDD','Has_SSD','Has_Flash_Storage']
categorical_features = ['Company', 'TypeName', 'Cpu_Brand', 'Gpu_Brand']

# Creating the Preprocessing Engine
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Building the Model Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])


In [36]:
# Train the model
model.fit(X_train, y_train)

#Test the model
y_pred = model.predict(X_test)
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R^2 Score:", r2_score(y_test, y_pred))

Mean Absolute Error: 10332.020970292875
R^2 Score: 0.8137009216285178


In [40]:
# this is one of the accuracy but we want is best model
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'regressor__n_estimators': [100, 200,300,400],
    'regressor__max_depth': [None, 20,30,40],
    'regressor__min_samples_split': [2, 5,10],
    'regressor__min_samples_leaf': [1, 2,4]
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator=model, param_distributions=param_grid, n_iter=15, cv=5, verbose=2, random_state=42, n_jobs=-1)